<img src="../../branding/aa-wordmark.png" alt="AA Engineering Toolkit" width="320px">

*Areeb Ahmed | Mechanical Engineer*

# Pressure Vessel Design
- **ASME Section VIII Div 1**
- Procedure
  - Design Pressure (internal or External), temperature, material
  - Vessel Geometry - Spherical>> Cylindrical as there are less corners
  - Wall thickness based on formulas
  - Head or end thickness
  - Nozzle reinforcement required or not?
  - Allowable stress and joint efficiency - manufacturing of welds important here
  - Corrosion and tolerance for longevity - based on process conditions
  - Check for buckling of thin walled pressure vessels under external pressure
  - Additional supports - saddle, skirt or legs: influence on local stress

**Internal pressure** and **external pressure** are handled slightly differently. Buckling is more important for external pressure (thickness-sizing) while stress-sizing is done for internal pressure. Stiffners are thuse more useful with external pressures.


## Worked example with `engtoolkit.asme`

A horizontal drum, sized end to end with the toolkit (the same calculations as the web app's
**Pressure vessel builder**):

- design pressure 1.0 MPa (g), full vacuum as external design condition, 150 °C
- inside diameter 1200 mm, tangent-to-tangent length 3000 mm
- carbon steel plate, allowable stress **S = 138 MPa** at design temperature: an assumed value,
  **check it against ASME II-D** for your material and temperature (the toolkit does not supply code tables)
- longitudinal seams spot radiographed, E = 0.85; seamless 2:1 ellipsoidal heads, E = 1.0
- corrosion allowance 3 mm
- one DN150 Sch 40 nozzle (ID 154.1 mm, wall 7.11 mm) in the shell, 300 × 10 mm reinforcing pad

> **Preliminary sizing only.** Verify every result against the current edition of ASME BPVC VIII-1.
> All quantities are SI inside the package (Pa, m); results are printed in MPa and mm here.

In [ ]:
from engtoolkit.asme.div1 import (
    JOINT_EFFICIENCIES, Nozzle, cylindrical_shell, external_pressure_cylinder, formed_head, hydrotest_pressure,
    nozzle_reinforcement, vessel_design,
)
from engtoolkit.materials import get_material

MPa, mm = 1e6, 1e-3
P, T = 1.0 * MPa, 423.15          # design pressure [Pa], design temperature [K]
D, L_tt = 1200 * mm, 3000 * mm    # inside diameter, tangent-to-tangent length [m]
S, E, CA = 138 * MPa, 0.85, 3 * mm
E_mod = get_material("Steel").prop("E")
print("Joint efficiency options (UW-12):")
for value, note in JOINT_EFFICIENCIES.items():
    print(f"  E = {value:.2f}: {note}")

### 1. Cylindrical shell (UG-27)

$t = \dfrac{PR}{SE - 0.6P}$ (circumferential), $t = \dfrac{PR}{2SE + 0.4P}$ (longitudinal), with R the corroded
inside radius; the thicker governs and CA is added. Leaving the nominal thickness blank picks the next standard plate.

In [ ]:
shell = cylindrical_shell(P, D, S, E, CA)
print(f"t circumferential {shell.t_circumferential / mm:.2f} mm, t longitudinal {shell.t_longitudinal / mm:.2f} mm")
print(f"required incl. CA {shell.t_required / mm:.2f} mm -> plate {shell.t_nominal / mm:.0f} mm, "
      f"MAWP {shell.mawp / MPa:.3f} MPa ({shell.clause})")

### 2. Heads (UG-32): comparing the options

In [ ]:
for kind in ("ellipsoidal_2_1", "torispherical", "hemispherical"):
    h = formed_head(P, D, S, kind, E=1.0, corrosion=CA)
    print(f"{kind:16s} t_req {h.t_required / mm:6.2f} mm  plate {h.t_nominal / mm:3.0f} mm  "
          f"MAWP {h.mawp / MPa:.3f} MPa  depth {h.depth / mm:5.0f} mm  ({h.clause})")

### 3. Nozzle reinforcement (UG-37, simplified)

Area replacement with F = 1, a set-in nozzle and outward fillet welds only: available area
$A_1 + A_2 + A_3 + A_{41} + A_{42} + A_5$ must be at least the required area $A = d\,t_r$.

In [ ]:
noz = nozzle_reinforcement(P, "cylinder", D, shell.t_nominal, S, d_nozzle=154.1 * mm, t_nozzle=7.11 * mm,
                           corrosion=CA, pad_diameter=300 * mm, pad_thickness=10 * mm,
                           weld_leg_nozzle=6 * mm, weld_leg_pad=6 * mm)
print(f"A required {noz.A_required / mm**2:.0f} mm², available {noz.A_available / mm**2:.0f} mm² -> {noz.status}")
print({k: round(getattr(noz, k) / mm**2) for k in ("A1", "A2", "A41", "A42", "A5")})

### 4. External pressure (UG-28), full vacuum

Factor A is estimated from the elastic buckling strain (the Section II-D Fig. G chart is not reproduced) and,
without a factor B from the material chart, the elastic formula $P_a = \dfrac{2AE}{3(D_o/t)}$ is used. Read B
from the II-D chart for your material and pass it as `B=` once A is known.

In [ ]:
Do = D + 2 * shell.t_nominal
L = L_tt + 2 * (D / 4) / 3          # UG-28(b): add 1/3 of each 2:1 head depth
ext = external_pressure_cylinder(101325, Do, L, shell.t_nominal, E_mod, corrosion=CA)
print(f"L/Do {ext.L_over_Do:.2f}, Do/t {ext.Do_over_t:.0f}, A = {ext.A:.3e} ({ext.A_source})")
print(f"Pa = {ext.P_allowable / 1e3:.1f} kPa -> {ext.status}; elastic t needed {ext.t_required_elastic / mm:.1f} mm")

### 5. Hydrostatic test (UG-99(b)) and the complete vessel

In [ ]:
print(f"Hydrotest at the shell MAWP: {hydrotest_pressure(shell.mawp, S, S).test_pressure / MPa:.3f} MPa")

vessel = vessel_design(P, D, S, length=L_tt, E=E, corrosion=CA, head_kind="ellipsoidal_2_1", P_external=101325,
                       E_modulus=E_mod, nozzles=[Nozzle("N1", d=154.1 * mm, t=7.11 * mm, pad_diameter=300 * mm,
                                                        pad_thickness=10 * mm, weld_leg=6 * mm, pad_weld_leg=6 * mm)])
from IPython.display import Markdown
Markdown(vessel.to_markdown())

In [ ]:
from engtoolkit import get_tool

tool = get_tool("asme.vessel_design")
inputs = tool.parse({"conditions": {"P": P, "T": T, "P_external": 101325}})
for chart in tool.make_charts(inputs, tool.run(inputs)):
    chart.to_figure()

## Background notes

| Component                         | Formula (Internal Pressure)                               | Notes                                                                                  |
| --------------------------------- | --------------------------------------------------------- | -------------------------------------------------------------------------------------- |
| **Cylindrical Shell (thin wall)** | $( t = \frac{P R}{S E - 0.6 P} + c ) $                      | R = inside radius, S = allowable stress, E = joint efficiency, c = corrosion allowance |
| **Spherical Shell**               | $( t = \frac{P R}{2 S E - 0.2 P} + c )  $                   | More efficient than cylinder; requires thinner wall                                    |
| **Hemispherical Head**            | $( t = \frac{P R}{2 S E - 0.2 P} + c )$                     | R = inside radius of head                                                              |
| **Ellipsoidal Head (2:1)**        | $( t = \frac{P D}{2 S E - 0.2 P} + c ) $                    | D = inside diameter of vessel (UG-32(d))                                               |
| **Torispherical Head**            | $( t = \frac{0.885 P L}{S E - 0.1 P} + c ) $                | L = inside crown radius; formula for knuckle radius = 6% of L (UG-32(e))               |
| **Minimum shell thickness**       | ASME VIII Div 1 Table U-1                                 | Usually ≥ 6 mm (¼ in) for carbon steel                                                 |
| **Nozzle reinforcement**          | $( A_r \geq A_n ) $(see ASME VIII App 2)                    | Required area of reinforcement ≥ nozzle area; includes welds and extra shell thickness |
| **Joint Efficiency (E)**          | 1.0 = full radiography, 0.85 = partial, 0.7 = fillet weld | Used in all thickness calculations                                                     |
| **Hydrostatic Test Pressure**     | $( P_{test} = 1.3 - 1.5 \times P_{design} )$                | Shell and heads must survive test                                                      |


## External Pressure (Vacuums)
- Buckling leading cause of failure thus $P_{cr}$ more important and the thickness is sized for this shell.
- UG-28/UG-31 Apply
- Heads UG-33 (Convex Pressure)

#### Formed Heads under Pressure on convex side UG-33
- Consideration of if welded or flanged (UG-35.1)
- Each head has it's own way of being calculated. Here only ellipsoidal head is considered.

**Procedure A (torispherical or ellipsoidal)**

1. $P_{internal}$ = 1.67 $P_{external}$
2. Then use UG-32 to get thickness via internal pressure with E=1

**Ellipsoidal Heads Procedure**

1.Assume t and get $\frac{R_o}{t}$
2. Get A
   $$ A = \frac{0.125}{\frac{R_o}{t}} $$
3. Use UG-28(d) Steps 2-6, basically get B from Material/temp Graph.
4. P_a
   $$ P_a = \frac{B}{\frac{R_o}{t}}$$
5. If A falls to the left
   $$ P_a = \frac{0.0625E}{(\frac{R_o}{t})^2}$$
6. Iterate until $P_{allow} > P_{design}$


#### UG-28 Thickness of shells and Tubes (Cylindrical Shells)

1.  $\frac{D_o}{t} > 10$ 
    1. Get Parameter A using this formula or *Figure G in subpart 3 of Section II, Part D. using $\frac{D_o}{L}$*
       $$ A= \frac{Et^3}{12(1-v^2)D_o}$$
    2. Get Factor B from the table provided for the material and service temperature
    3. Get $P_{allow}$ from
       $$P_a= \frac{4B}{3(\frac{D_o}{t})} $$
    4. If A falls left of material/temperature line
       $$P_a= \frac{2AE}{3(\frac{D_o}{t})} $$
    5. Use charts in UG-27/UG-30 to get radius to thickness ratio. Ensure L falls within chart limits
    6. Check head (UG-31)
    7. Iterate until $P_{allow} > P_{design}$
 2. $\frac{D_o}{t} < 10$ 
    1. A calculated as 
       $$A= \frac{1.1}{(\frac{D_o}{t})^2}$$
    2. For A> 0.1 A= 0.1
    3. Get B in the same way as above
    4. Get $P_{a1}$ $$P_{a2} =[\frac{2.167}{\frac{D_o}{t}}-0.0833]B $$
    5. Get $P_{a2}$  $$P_{a2} =\frac{2S}{\frac{D_o}{t}}[1-\frac{1}{\frac{D_o}{t}}] $$
    6. Where S is $ min(2\sigma_{a,tension},0.9\sigma_y)$ from UG-23 and $\sigma_y = 2B_{RHS termination}$ respectively
    7. Use smaller of $P_{a1}$ and $P_{a2}$
    8. Iterate until $P_{allow} > P_{design}$

#### UG-29 Stiffening Rings for Cylindrical shells under external Presuure
- Usually when the L is quite long
- Attached via welding or brazing
- Attachment requirements- welding, MoI requirements, internal vs external

### Reinforcement Required for Openings
- small openings
- Openings in flat heads
- spacing between 2 openings - ligament width, c2c spacing

### Nozzle Neck Thickness
- Size chart that shows for each DN what the Nozzle thickness should be (Table UG-45)
- Inspection openings - manhole or handhole

#### Supports
- UG-54

### Lugs for Platforms, Ladders and Lifting Points
- UG-55